# Expectation, Variance & Moments

## What's covered

- **Expectation** — the average value of a random variable; weighted by probability
- **Variance** and **standard deviation** — how spread out a distribution is
- **Higher moments** — skewness (asymmetry) and kurtosis (tail weight)
- **Covariance** and **correlation** — measuring relationships between two random variables
- **Linearity of expectation** and its surprisingly powerful consequences
- **LOTUS** — the law of the unconscious statistician
- **Variance of sums** — and the role of correlation
- Bias-variance decomposition and why these identities matter in ML


## Expectation

The **expectation** (or **expected value**, or **mean**) of a random variable `X` is the long-run average value if you sampled `X` infinitely many times. Notationally `\mathbb{E}[X]` or `\mu_X`.

**Discrete case.** Weight each possible value by its probability and sum:

$$
\mathbb{E}[X] = \sum_x x \, p_X(x)
$$

**Continuous case.** Same idea, replacing sum with integral:

$$
\mathbb{E}[X] = \int x \, f_X(x) \, dx
$$

This is the same integral we met in the calculus repo — expectation is integration against a density. The mass-weighted center of the distribution. The point where, if the PDF were a physical mass distribution, the curve would balance.

**Common expectations to know.**

| Distribution | `E[X]` |
|---|---|
| `\text{Bernoulli}(p)` | `p` |
| `\text{Binomial}(n, p)` | `np` |
| `\text{Poisson}(\lambda)` | `\lambda` |
| `\text{Geometric}(p)` | `1/p` |
| `\mathcal{U}(a, b)` | `(a + b)/2` |
| `\text{Exp}(\lambda)` | `1/\lambda` |
| `\mathcal{N}(\mu, \sigma^2)` | `\mu` |
| `\text{Beta}(\alpha, \beta)` | `\alpha / (\alpha + \beta)` |

These are worth memorizing — they show up constantly. The variance table comes next.

**The estimator.** Given samples `x_1, x_2, \dots, x_n`, the **sample mean** `\bar{x} = (1/n) \sum_i x_i` is an unbiased estimator of `\mathbb{E}[X]`. The Law of Large Numbers (notebook 8) says `\bar{x} \to \mathbb{E}[X]` as `n \to \infty`.


In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)

# Verify the mean formulas for a few distributions
N = 200_000

samples = {
    "Bernoulli(0.3)":  rng.random(N) < 0.3,
    "Binomial(20, 0.3)": rng.binomial(20, 0.3, N),
    "Poisson(4)":      rng.poisson(4, N),
    "Geometric(0.25)": rng.geometric(0.25, N),
    "Uniform(2, 8)":   rng.uniform(2, 8, N),
    "Exp(rate=2)":     rng.exponential(scale=1/2, size=N),  # numpy's scale = 1/rate
    "N(5, 4)":         rng.normal(5, 2, N),                  # std = 2 → var = 4
}
analytic = {
    "Bernoulli(0.3)": 0.3,
    "Binomial(20, 0.3)": 20 * 0.3,
    "Poisson(4)": 4,
    "Geometric(0.25)": 1/0.25,
    "Uniform(2, 8)": (2 + 8) / 2,
    "Exp(rate=2)": 1/2,
    "N(5, 4)": 5,
}

print(f"{'distribution':<20} {'sample E[X]':>14} {'analytic E[X]':>15}")
for name, s in samples.items():
    print(f"{name:<20} {s.mean():>14.4f} {analytic[name]:>15.4f}")


## Linearity of expectation

The single most useful identity in probability:

$$
\mathbb{E}[aX + bY + c] = a \, \mathbb{E}[X] + b \, \mathbb{E}[Y] + c
$$

This holds for **any** random variables `X` and `Y` — they don't have to be independent, they don't have to be from the same distribution, they don't have to have anything in common at all. Linearity is *unconditional*. It's the reason so many "obviously hard" problems have one-line solutions in probability.

**Two consequences.**

- `\mathbb{E}[X + Y] = \mathbb{E}[X] + \mathbb{E}[Y]` for any `X, Y`.
- `\mathbb{E}[cX] = c \, \mathbb{E}[X]` for any constant `c`.

**A classic application — expected number of fixed points.** Permute `n` letters at random; how many letters land back in their original position on average? Define `X_i = 1` if letter `i` is fixed, else 0. Each `X_i \sim \text{Bernoulli}(1/n)`. Total fixed points = `X_1 + X_2 + \dots + X_n`. By linearity:

$$
\mathbb{E}\left[\sum_i X_i\right] = \sum_i \mathbb{E}[X_i] = n \cdot \frac{1}{n} = 1
$$

The expected number is exactly 1, regardless of `n`. No combinatorial computation needed.

**ML angle.** The training loss is an expectation over the dataset. By linearity, `\nabla_w \mathbb{E}[L_i(w)] = \mathbb{E}[\nabla_w L_i(w)]` — you can compute the gradient of the average by averaging the per-sample gradients. *That's why SGD works at all.*


## LOTUS — the law of the unconscious statistician

If `Y = g(X)` for some function `g`, you can compute `\mathbb{E}[Y]` *without* first deriving the distribution of `Y`. Just plug `g(x)` into the expectation formula for `X`:

$$
\mathbb{E}[g(X)] = \sum_x g(x) \, p_X(x) \qquad \text{(discrete)}
$$
$$
\mathbb{E}[g(X)] = \int g(x) \, f_X(x) \, dx \qquad \text{(continuous)}
$$

This is **LOTUS** — the *Law of the Unconscious Statistician*. The name is a joke: statisticians use it without explicit justification because it's so natural.

LOTUS is why we can compute, say, `\mathbb{E}[X^2]` directly from the distribution of `X` rather than having to derive the distribution of `X^2`. It's the workhorse behind variance, moments, and moment-generating functions.

**Beware:** `\mathbb{E}[g(X)] \neq g(\mathbb{E}[X])` in general. This is a common student mistake. If `g` is non-linear, you cannot just push the expectation inside. **Jensen's inequality** (notebook 7) tells you the direction of the gap: `\mathbb{E}[g(X)] \geq g(\mathbb{E}[X])` for convex `g`, opposite for concave. The reason `\log \mathbb{E}[\cdot] \neq \mathbb{E}[\log \cdot]` causes endless confusion in variational inference.


In [ ]:
# E[X^2] for X ~ N(0, 1) — true value is 1 (since Var(X) = 1, E[X] = 0)
# By LOTUS, E[X^2] = integral of x^2 * f(x) dx — or just average X^2 over samples
N = 500_000
samples = rng.normal(0, 1, N)

print(f"E[X^2] (sample)   = {(samples ** 2).mean():.4f}     (analytic 1.0)")
print(f"E[X^4] (sample)   = {(samples ** 4).mean():.4f}     (analytic 3.0 — see kurtosis)")
print(f"E[e^X] (sample)   = {np.exp(samples).mean():.4f}     (analytic e^(1/2) = {np.exp(0.5):.4f})")
print()
print(f"g(E[X])  for g(x) = x^2:  g(0) = 0")
print(f"E[g(X)]                :        {(samples**2).mean():.4f}")
print("Notice they DISAGREE — non-linear g blocks pushing E inside.")


## Variance and standard deviation

Expectation gives you a center. **Variance** measures how far values typically stray from that center:

$$
\text{Var}(X) = \mathbb{E}\bigl[(X - \mu)^2\bigr] = \mathbb{E}[X^2] - \mu^2
$$

Both forms are useful. The first is the geometric definition (average squared distance from the mean). The second is the **computational formula** — often easier in practice. Always non-negative.

**Standard deviation** `\sigma_X = \sqrt{\text{Var}(X)}` is in the same units as `X`, which is why it's often reported alongside the mean.

**Common variances to know.**

| Distribution | `Var(X)` |
|---|---|
| `\text{Bernoulli}(p)` | `p(1 - p)` |
| `\text{Binomial}(n, p)` | `np(1 - p)` |
| `\text{Poisson}(\lambda)` | `\lambda` |
| `\text{Geometric}(p)` | `(1 - p)/p^2` |
| `\mathcal{U}(a, b)` | `(b - a)^2 / 12` |
| `\text{Exp}(\lambda)` | `1/\lambda^2` |
| `\mathcal{N}(\mu, \sigma^2)` | `\sigma^2` |
| `\text{Beta}(\alpha, \beta)` | `\alpha\beta / [(\alpha+\beta)^2 (\alpha+\beta+1)]` |

**Useful identities.** `\text{Var}(c) = 0`, `\text{Var}(aX + b) = a^2 \text{Var}(X)`. The shift `b` doesn't affect spread; the scale `a` enters squared because variance is in squared units.

**Sample variance.** Given data `x_1, \dots, x_n`, the standard estimator is

$$
s^2 = \frac{1}{n - 1} \sum_{i=1}^{n} (x_i - \bar{x})^2
$$

Note the **`n - 1`** denominator (Bessel's correction). Using `n` would *underestimate* the true variance because we're already using `\bar{x}` (which "absorbs" some of the variability). `n - 1` makes the estimator unbiased. NumPy's `np.var` uses `n` by default; `np.var(..., ddof=1)` uses `n - 1`.


In [ ]:
# Variance check
N = 200_000

print(f"{'distribution':<20} {'sample Var':>12} {'analytic Var':>13}")
checks = [
    ("Bernoulli(0.3)",  rng.random(N) < 0.3,                0.3 * 0.7),
    ("Binomial(20, 0.3)", rng.binomial(20, 0.3, N),         20 * 0.3 * 0.7),
    ("Poisson(4)",      rng.poisson(4, N),                  4),
    ("Uniform(2, 8)",   rng.uniform(2, 8, N),               (8 - 2)**2 / 12),
    ("Exp(rate=2)",     rng.exponential(scale=0.5, size=N), (1/2)**2),
    ("N(5, 4)",         rng.normal(5, 2, N),                4),
]
for name, s, an in checks:
    print(f"{name:<20} {s.var(ddof=1):>12.4f} {an:>13.4f}")


## Higher moments — skewness and kurtosis

`\mathbb{E}[X]` is the **first moment**. `\mathbb{E}[X^2]` is the **second moment**. Higher moments tell you more about the shape of a distribution beyond center and spread.

**Skewness** measures asymmetry — how lopsided the distribution is:

$$
\text{Skew}(X) = \mathbb{E}\!\left[\left(\frac{X - \mu}{\sigma}\right)^3\right]
$$

- **Skew = 0** — symmetric (Gaussian, Uniform, t-distribution).
- **Skew > 0** — right tail is longer (Exponential, Log-normal, salary distributions).
- **Skew < 0** — left tail is longer (rarer; some performance scores like ages-at-death).

**Kurtosis** measures tail weight — how heavy the tails are relative to a Gaussian:

$$
\text{Kurt}(X) = \mathbb{E}\!\left[\left(\frac{X - \mu}{\sigma}\right)^4\right]
$$

Excess kurtosis = `Kurt(X) - 3` is the more common quantity (Gaussian has kurtosis exactly 3).

- **Excess kurt > 0** — heavy tails, more extreme outliers than Gaussian (Cauchy, t-distribution with low DoF, financial returns).
- **Excess kurt < 0** — light tails, very few outliers (Uniform).

**ML angle.** Sub-Gaussian and heavy-tailed assumptions appear in modern theory: gradient noise in SGD has been shown to be heavy-tailed, which affects convergence behavior. PCA assumes Gaussian-like data and is sensitive to outliers — which kurtosis measures.


In [ ]:
# Inspect skewness and kurtosis of three distributions
N = 500_000
checks = {
    "Normal":      rng.normal(0, 1, N),
    "Exponential": rng.exponential(1, N),
    "Uniform":     rng.uniform(-1, 1, N),
}

print(f"{'distribution':<14} {'mean':>8} {'std':>8} {'skew':>8} {'ex.kurt':>10}")
for name, s in checks.items():
    print(f"{name:<14} {s.mean():>8.3f} {s.std():>8.3f} {stats.skew(s):>8.3f} {stats.kurtosis(s):>10.3f}")

print("\nNormal: skew ≈ 0, ex.kurt ≈ 0 (by definition).")
print("Exponential: skew = 2, ex.kurt = 6 — right-skewed, heavy-tailed.")
print("Uniform: skew ≈ 0, ex.kurt ≈ -1.2 — symmetric but light-tailed.")


## Covariance and correlation

For two random variables `X` and `Y`, **covariance** measures their joint variability:

$$
\text{Cov}(X, Y) = \mathbb{E}[(X - \mu_X)(Y - \mu_Y)] = \mathbb{E}[XY] - \mu_X \mu_Y
$$

Sign tells you the direction of association: positive (both move up together), negative (one up, the other down), zero (no *linear* association). Magnitude depends on the units of `X` and `Y`, which is annoying.

**Correlation** is covariance rescaled to be unit-free:

$$
\text{Corr}(X, Y) = \rho_{XY} = \frac{\text{Cov}(X, Y)}{\sigma_X \sigma_Y}
$$

Always between `-1` and `1`. Interpretable as: "how aligned are `X` and `Y` after standardizing them?" Same direction as covariance, but bounded.

**Two crucial facts.**

- `\text{Var}(X) = \text{Cov}(X, X)`. Variance is self-covariance.
- **Independence implies zero covariance**, but **zero covariance does NOT imply independence**. Covariance only sees *linear* dependence. `X \sim \mathcal{U}(-1, 1)` and `Y = X^2` are perfectly determined by each other, but their covariance is zero.

**Covariance matrix** (multivariate). For `\mathbf{X} = (X_1, \dots, X_d)`, the covariance matrix `\Sigma` is the `d \times d` matrix with entries `\Sigma_{ij} = \text{Cov}(X_i, X_j)`. Always symmetric and PSD (positive semi-definite — eigenvalues `\geq 0`). This is the object PCA decomposes.


In [ ]:
# Covariance and correlation — two positively correlated normals, plus an XOR-style example
N = 50_000

# Two jointly normal variables with correlation 0.7
mean = [0, 0]
cov_target = [[1.0, 0.7], [0.7, 1.0]]
joint = rng.multivariate_normal(mean, cov_target, size=N)
X, Y = joint[:, 0], joint[:, 1]

print(f"Cov(X, Y) ~ {np.cov(X, Y, ddof=1)[0, 1]:.4f}   (target 0.7)")
print(f"Corr(X, Y) ~ {np.corrcoef(X, Y)[0, 1]:.4f}   (target 0.7)")

# Now: X ~ Uniform(-1, 1), Y = X^2 — perfectly determined, but cov ≈ 0
X = rng.uniform(-1, 1, N)
Y = X ** 2
print(f"\nFor Y = X^2 (X ~ Uniform(-1, 1)):")
print(f"  Cov(X, Y) ~ {np.cov(X, Y, ddof=1)[0, 1]:.4f}   (analytic 0)")
print(f"  Corr(X, Y) ~ {np.corrcoef(X, Y)[0, 1]:.4f}   (analytic 0)")
print("  X and Y are completely dependent but linearly uncorrelated.")


## Variance of a sum

This identity is everywhere in ML — bias-variance, ensemble methods, sample-mean variance:

$$
\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y) + 2 \, \text{Cov}(X, Y)
$$

**Special case — independence.** If `X` and `Y` are independent, `\text{Cov}(X, Y) = 0` and the formula collapses to

$$
\text{Var}(X + Y) = \text{Var}(X) + \text{Var}(Y) \qquad \text{(independent)}
$$

For `n` independent random variables, this generalizes to `\text{Var}(\sum X_i) = \sum \text{Var}(X_i)`.

**The sample mean.** If `X_1, \dots, X_n` are i.i.d. with variance `\sigma^2`, then `\bar{X}_n = (1/n) \sum X_i` has

$$
\text{Var}(\bar{X}_n) = \frac{\sigma^2}{n}
$$

so `\text{SD}(\bar{X}_n) = \sigma / \sqrt{n}`. **This is the `1/\sqrt{n}` rate** that shows up in Monte Carlo error, in CLT bounds, in standard errors of regression coefficients — every "average more samples to reduce noise" argument in ML. Doubling samples doesn't double precision; quadrupling samples does.

**Bias-variance decomposition** (preview, made formal in notebook 6). For a regression estimator `\hat{f}`,

$$
\text{MSE}(\hat{f}(x)) = \bigl(\text{Bias}(\hat{f}(x))\bigr)^2 + \text{Var}(\hat{f}(x)) + \sigma_{\text{noise}}^2
$$

The decomposition explains why simple models often beat complex ones on small data — variance dominates when you have too few samples to pin down the model precisely.


In [ ]:
# Verify Var(sample mean) = sigma^2 / n
sigma2 = 4.0
N_replicas = 10_000

print(f"{'n':>6} {'Var(mean) sample':>20} {'sigma^2 / n':>14}")
for n in [10, 100, 1_000, 10_000]:
    means = np.array([rng.normal(0, np.sqrt(sigma2), n).mean() for _ in range(N_replicas)])
    print(f"{n:>6} {means.var(ddof=1):>20.6f} {sigma2 / n:>14.6f}")

# Variance of sum is sum of variances when independent
X = rng.normal(0, 1, 100_000)
Y = rng.normal(0, 1, 100_000)
print(f"\nVar(X) = {X.var():.4f}, Var(Y) = {Y.var():.4f}")
print(f"Var(X + Y) = {(X + Y).var():.4f}  (analytic ≈ 1 + 1 = 2)")


## Where this appears in ML

Expectation, variance, and the related identities are the load-bearing math behind most "what's the expected loss / expected reward / expected accuracy" arguments in ML.

- **Empirical risk minimization.** The loss `L(w) = \mathbb{E}_{(x, y) \sim \text{data}}[\ell(w; x, y)]` is an expectation; SGD estimates its gradient via a sample mean — and that sample mean's variance is exactly `\sigma^2 / B` for batch size `B`. Doubling batch size halves variance.
- **Bias-variance decomposition.** `\mathbb{E}[(\hat{f} - f)^2] = \text{Bias}^2 + \text{Var}(\hat{f}) + \sigma^2_{\text{noise}}`. Explains overfitting, regularization, and the value of more data.
- **Bagging / ensembles.** Average `k` independent models. Each has variance `\sigma^2`; the ensemble has variance `\sigma^2 / k` (linearity + independence). Random forests literally exploit this identity.
- **Boosting.** Reduce *bias* by sequential refinement instead of variance — the opposite trade-off.
- **Linearity of gradient = backprop math.** `\nabla \mathbb{E}[L_i] = \mathbb{E}[\nabla L_i]`. Mini-batch gradients are unbiased estimators of the true gradient because of linearity of expectation.
- **Reward shaping in RL.** The expected return `\mathbb{E}[\sum_t \gamma^t r_t]` is a sum; you can transform reward functions in ways that preserve gradient direction (REINFORCE baselines exploit this).
- **Control variates and Rao-Blackwellization.** Variance reduction techniques in MC estimation lean on covariance and conditional expectation identities.
- **Standard errors and confidence intervals.** Width of a confidence interval shrinks like `\sigma / \sqrt{n}` — direct consequence of the variance-of-sample-mean identity.
- **Calibration and Brier scores.** Calibration losses are squared deviations between predicted probabilities and outcomes — second-moment objects.
- **GMM / EM / VAEs.** All use expectation-style E-steps to compute responsibility-weighted means.
- **Score functions and REINFORCE.** `\nabla \mathbb{E}_\theta[R] = \mathbb{E}_\theta[R \nabla \log \pi_\theta]` — a moment-style identity central to policy gradients.

Next notebook: **joint, marginal, and conditional distributions** — when more than one random variable matters, and the elegant arithmetic that handles them.
